<a href="https://colab.research.google.com/github/Jacksonwsmith/TrilogyOCR_Pipeline/blob/main/Mistral_Jackson_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mistralai pandas pymupdf pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.3/509.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 9.3 MB/s eta 0:00:00


In [ ]:
from google.colab import files
import os

# Create checks folder
os.makedirs('checks', exist_ok=True)

# Upload files
print("📂 Click 'Choose Files' and select your PDF check files...")
uploaded = files.upload()

# Move files to checks folder
for filename in uploaded.keys():
    os.rename(filename, f'checks/{filename}')
    print(f"   ✓ Uploaded {filename}")

# Verify files are there
check_files = [f for f in os.listdir('checks') if f.endswith('.pdf')]
print(f"\n✅ Found {len(check_files)} PDF file(s) in checks folder")

📂 Click 'Choose Files' and select your PDF check files...


Saving WJF-DIVERSIFIED-20260127-$1031.33-5001502951.pdf to WJF-DIVERSIFIED-20260127-$1031.33-5001502951.pdf
   ✓ Uploaded WJF-DIVERSIFIED-20260127-$1031.33-5001502951.pdf

✅ Found 1 PDF file(s) in checks folder


In [ ]:
import os
import io
import csv
import json
import base64
import logging
import time
import re
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import fitz  # PyMuPDF
from PIL import Image
from mistralai import Mistral

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "") # Use the kernel state value as default

# ----------------------------
# Config
# ----------------------------
MODEL = os.getenv("MISTRAL_MODEL", "pixtral-large-latest")
DPI = int(os.getenv("PDF_RENDER_DPI", "220"))          # 200-300 is usually enough
MAX_TOKENS = int(os.getenv("MISTRAL_MAX_TOKENS", "30000"))
MAX_RETRIES = 1
RETRY_DELAY_SECONDS = 2

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s"
)

# ----------------------------
# Prompt
# ----------------------------
PROMPT = """You are extracting data from an oil & gas royalty check statement. Extract EVERY production detail line.\n\nFor EACH line in the revenue statement table, create a separate JSON object in an array.\n\nEach object should have these EXACT field names (use empty string "" for missing values):\n{\n  "Operator_ID": "",\n  "Operator_Name": "DIVERSIFIED",\n  "Owner_Name": "Wilson Johnson Family",\n  "Owner_Number": "1064941",\n  "Check_Number": "5001502951",\n  "Check_Date": "1/27/2026",\n  "Check_Amount": "1031.33",\n  "Operator_CC": "1119847.01",\n  "Operator_API": "4703305324",\n  "Partner_API": "",\n  "MA_API": "",\n  "Partner_CC": "",\n  "Property_Description": "FORTNEY D 156, TOWN/DISTRICT: 04 EAGLE",\n  "Property_State": "WV",\n  "Property_County": "HARRISON",\n  "Product_Code": "",\n  "Product_Description": "Gas",\n  "Interest_Code": "",\n  "Interest_Type": "Royalty Interest",\n  "Owner_Percent": "0.0625",\n  "Distribution_Percent": "0.0625",\n  "Prod_Date": "11/1/2025",\n  "Price": "2.63",\n  "BTU_Factor": "1.0002",\n  "Gross_Volume": "605.27",\n  "Gross_Value": "1593.71",\n  "Gross_Taxes": "0",\n  "Gross_Deducts": "0",\n  "Net_Value": "1593.71",\n  "Owner_Gross_Volume": "37.83",\n  "Owner_Gross_Value": "99.61",\n  "Owner_Gross_Taxes": "0",\n  "Owner_Gross_Deducts": "0",\n  "Owner_Net_Value": "99.61",\n  "Tax_Code_1": "", "Tax_Type_1": "", "Gross_Tax_1": "", "Net_Tax_1": "",\n  "Tax_Code_2": "", "Tax_Type_2": "", "Gross_Tax_2": "", "Net_Tax_2": "",\n  "Tax_Code_3": "", "Tax_Type_3": "", "Gross_Tax_3": "", "Net_Tax_3": "",\n  "Tax_Code_4": "", "Tax_Type_4": "", "Gross_Tax_4": "", "Net_Tax_4": "",\n  "Tax_Code_5": "", "Tax_Type_5": "", "Gross_Tax_5": "", "Net_Tax_5": "",\n  "Tax_Code_6": "", "Tax_Type_6": "", "Gross_Tax_6": "", "Net_Tax_6": "",\n  "Tax_Code_7": "", "Tax_Type_7": "", "Gross_Tax_7": "", "Net_Tax_7": "",\n  "Tax_Code_8": "", "Tax_Type_8": "", "Gross_Tax_8": "", "Net_Tax_8": "",\n  "Tax_Code_9": "", "Tax_Type_9": "", "Gross_Tax_9": "", "Net_Tax_9": "",\n  "Tax_Code_10": "", "Tax_Type_10": "", "Gross_Tax_10": "", "Net_Tax_10": "",\n  "Deduct_Code_1": "", "Deduct_Type_1": "", "Gross_Deduct_1": "", "Net_Deduct_1": "",\n  "Deduct_Code_2": "", "Deduct_Type_2": "", "Gross_Deduct_2": "", "Net_Deduct_2": "",\n  "Deduct_Code_3": "", "Deduct_Type_3": "", "Gross_Deduct_3": "", "Net_Deduct_3": "",\n  "Deduct_Code_4": "", "Deduct_Type_4": "", "Gross_Deduct_4": "", "Net_Deduct_4": "",\n  "Deduct_Code_5": "", "Deduct_Type_5": "", "Gross_Deduct_5": "", "Net_Deduct_5": "",\n  "Deduct_Code_6": "", "Deduct_Type_6": "", "Gross_Deduct_6": "", "Net_Deduct_6": "",\n  "Deduct_Code_7": "", "Deduct_Type_7": "", "Gross_Deduct_7": "", "Net_Deduct_7": "",\n  "Deduct_Code_8": "", "Deduct_Type_8": "", "Gross_Deduct_8": "", "Net_Deduct_8": "",\n  "Deduct_Code_9": "", "Deduct_Type_9": "", "Gross_Deduct_9": "", "Net_Deduct_9": "",\n  "Deduct_Code_10": "", "Deduct_Type_10": "", "Gross_Deduct_10": "", "Net_Deduct_10": "",\n  "Detail_Line_Notation": ""\n}\n\nCRITICAL INSTRUCTIONS:\n- Operator_CC = the Property number (e.g., "1119847.01")\n- Operator_API = the "Operator API#" number (e.g., "4703305324")\n- Product_Description = "Gas" or "Oil" from the Type column\n- Interest_Type = "Royalty Interest" or "FRR1" from the Type column\n- Create ONE object per line in the revenue table\n- Return a JSON ARRAY with ALL detail lines found on this page\n- Use empty strings "" for any missing data\n- Remove commas from numbers (e.g., "1,593.71" becomes "1593.71")\n- Format dates as M/D/YYYY (e.g., "11/1/2025")\n\nReturn ONLY the JSON array, no markdown, no explanation.\n"""

# ----------------------------
# CSV columns + mapping
# ----------------------------
CSV_COLUMNS = [
    "Operator ID", "Operator Name", "Owner Name", "Owner Number",
    "Check Number", "Check Date", "Check Amount", "Operator CC",
    "Operator API", "Partner API", "MA API", "Partner CC",
    "Property Description", "Property State", "Property County",
    "Product Code", "Product Description", "Interest Code", "Interest Type",
    "Owner Percent", "Distribution Percent", "Prod Date", "Price",
    "BTU Factor", "Gross Volume", "Gross Value", "Gross Taxes",
    "Gross Deducts", "Net Value", "Owner Gross Volume", "Owner Gross Value",
    "Owner Gross Taxes", "Owner Gross Deducts", "Owner Net Value",
]

for i in range(1, 11):
    CSV_COLUMNS += [f"Tax Code {i}", f"Tax Type {i}", f"Gross Tax {i}", f"Net Tax {i}"]
for i in range(1, 11):
    CSV_COLUMNS += [f"Deduct Code {i}", f"Deduct Type {i}", f"Gross Deduct {i}", f"Net Deduct {i}"]

CSV_COLUMNS += ["Detail Line Notation"]

JSON_TO_CSV = {
    "Operator_ID": "Operator ID",
    "Operator_Name": "Operator Name",
    "Owner_Name": "Owner Name",
    "Owner_Number": "Owner Number",
    "Check_Number": "Check Number",
    "Check_Date": "Check Date",
    "Check_Amount": "Check Amount",
    "Operator_CC": "Operator CC",
    "Operator_API": "Operator API",
    "Partner_API": "Partner API",
    "MA_API": "MA API",
    "Partner_CC": "Partner CC",
    "Property_Description": "Property Description",
    "Property_State": "Property State",
    "Property_County": "Property County",
    "Product_Code": "Product Code",
    "Product_Description": "Product Description",
    "Interest_Code": "Interest Code",
    "Interest_Type": "Interest Type",
    "Owner_Percent": "Owner Percent",
    "Distribution_Percent": "Distribution Percent",
    "Prod_Date": "Prod Date",
    "Price": "Price",
    "BTU_Factor": "BTU Factor",
    "Gross_Volume": "Gross Volume",
    "Gross_Value": "Gross Value",
    "Gross_Taxes": "Gross Taxes",
    "Gross_Deducts": "Gross Deducts",
    "Net_Value": "Net Value",
    "Owner_Gross_Volume": "Owner Gross Volume",
    "Owner_Gross_Value": "Owner Gross Value",
    "Owner_Gross_Taxes": "Owner Gross Taxes",
    "Owner_Gross_Deducts": "Owner Gross Deducts",
    "Owner_Net_Value": "Owner Net Value",
    "Detail_Line_Notation": "Detail Line Notation",
}

# ----------------------------
# Helpers
# ----------------------------
def require_api_key() -> str:
    key = MISTRAL_API_KEY.strip() # Directly use the Python variable
    if not key:
        raise RuntimeError(
            "Missing MISTRAL_API_KEY. Set the Python variable `MISTRAL_API_KEY` or environment variable.\n"
            "  export MISTRAL_API_KEY='...'\n"
            "Or define it as a string at the top of the cell."
        )
    return key

def pdf_page_to_base64_jpeg(doc: fitz.Document, page_num: int, dpi: int = 220) -> Tuple[str, str]:
    page = doc[page_num]
    pix = page.get_pixmap(dpi=max(72, min(dpi, 350)))  # clamp
    mode = "RGB"
    # pix.alpha indicates an alpha channel; if present, convert to RGB safely
    img = Image.frombytes("RGB" if pix.alpha == 0 else "RGBA", [pix.width, pix.height], pix.samples)
    if img.mode != mode:
        img = img.convert(mode)

    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=92, optimize=True)
    return base64.b64encode(buf.getvalue()).decode("utf-8"), "image/jpeg"

def strip_code_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        lines = t.splitlines()
        # drop first and last fence lines
        lines = lines[1:-1]
        t = "\n".join(lines).strip()
        if t.lower().startswith("json"):
            t = t[4:].strip()
    return t

def parse_json_array_loose(text: str) -> List[Dict[str, Any]]:
    """
    Tries to parse:
    - pure JSON array
    - JSON wrapped in fences
    - model that accidentally adds leading/trailing text
    """
    raw = strip_code_fences(text)

    # Fast path
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            return [parsed]
        if isinstance(parsed, list):
            return [x for x in parsed if isinstance(x, dict)]
        return []
    except Exception:
        pass

    # Loose path: extract first [...] region
    start = raw.find("[")
    end = raw.rfind("]")
    if start != -1 and end != -1 and end > start:
        candidate = raw[start : end + 1]
        parsed = json.loads(candidate)
        if isinstance(parsed, list):
            return [x for x in parsed if isinstance(x, dict)]

    raise ValueError("Could not parse JSON array from model output.")

def json_row_to_csv_row(item: Dict[str, Any]) -> Dict[str, str]:
    row = {col: "" for col in CSV_COLUMNS}

    # direct fields
    for jkey, ckey in JSON_TO_CSV.items():
        row[ckey] = str(item.get(jkey, "") if item.get(jkey, "") is not None else "")

    # taxes
    for i in range(1, 11):
        row[f"Tax Code {i}"] = str(item.get(f"Tax_Code_{i}", "") if item.get(f"Tax_Code_{i}", "") is not None else "")
        row[f"Tax Type {i}"] = str(item.get(f"Tax_Type_{i}", "") if item.get(f"Tax_Type_{i}", "") is not None else "")
        row[f"Gross Tax {i}"] = str(item.get(f"Gross_Tax_{i}", "") if item.get(f"Gross_Tax_{i}", "") is not None else "")
        row[f"Net Tax {i}"] = str(item.get(f"Net_Tax_{i}", "") if item.get(f"Net_Tax_{i}", "") is not None else "")

    # deducts
    for i in range(1, 11):
        row[f"Deduct Code {i}"] = str(item.get(f"Deduct_Code_{i}", "") if item.get(f"Deduct_Code_{i}", "") is not None else "")
        row[f"Deduct Type {i}"] = str(item.get(f"Deduct_Type_{i}", "") if item.get(f"Deduct_Type_{i}", "") is not None else "")
        row[f"Gross Deduct {i}"] = str(item.get(f"Gross_Deduct_{i}", "") if item.get(f"Gross_Deduct_{i}", "") is not None else "")
        row[f"Net Deduct {i}"] = str(item.get(f"Net_Deduct_{i}", "") if item.get(f"Net_Deduct_{i}", "") is not None else "")

    return row

def ask_model_for_page(client: Mistral, image_b64: str, media_type: str) -> List[Dict[str, Any]]:
    last_error: Exception | None = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.complete(
                model=MODEL,
                temperature=0,
                max_tokens=MAX_TOKENS,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "image_url", "image_url": f"data:{media_type};base64,{image_b64}"},
                            {"type": "text", "text": PROMPT},
                        ],
                    }
                ],
            )
            content = response.choices[0].message.content
            return parse_json_array_loose(content)
        except Exception as exc:
            last_error = exc
            logging.warning(f"Attempt {attempt} failed: {exc}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY_SECONDS * attempt)
    raise RuntimeError(f"Model call failed after {MAX_RETRIES} attempts: {last_error}")

def process_checks_to_csv(pdf_folder: str, output_csv: str) -> None:
    pdf_dir = Path(pdf_folder)
    if not pdf_dir.exists() or not pdf_dir.is_dir():
        raise FileNotFoundError(f"PDF folder does not exist: {pdf_folder}")

    pdf_paths = sorted([p for p in pdf_dir.iterdir() if p.suffix.lower() == ".pdf"])
    if not pdf_paths:
        print("No PDF files found.")
        return

    api_key = require_api_key()
    client = Mistral(api_key=api_key)

    total_rows = 0
    print(f"Found {len(pdf_paths)} PDF file(s). Processing...")

    with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=CSV_COLUMNS)
        writer.writeheader()

        for pdf_path in pdf_paths:
            print(f"Processing {pdf_path.name}")
            file_rows = 0

            try:
                doc = fitz.open(pdf_path)
                for page_num in range(doc.page_count):
                    image_b64, media_type = pdf_page_to_base64_jpeg(doc, page_num, dpi=DPI)
                    detail_lines = ask_model_for_page(client, image_b64, media_type)
                    for line in detail_lines:
                        writer.writerow(json_row_to_csv_row(line))
                        file_rows += 1
                        total_rows += 1
                    print(f"  Page {page_num + 1}: {len(detail_lines)} detail line(s)")
                doc.close()

                print(f"  Done {pdf_path.name}: {file_rows} rows")

            except Exception as exc:
                logging.error(f"  ERROR in {pdf_path.name}: {exc}", exc_info=True)

    print(f"Done. Wrote {total_rows} row(s) to {output_csv}")

if __name__ == "__main__":
    process_checks_to_csv(
        pdf_folder="./checks",
        output_csv="royalty_checks.csv",
    )

Found 1 PDF file(s). Processing...
Processing WJF-DIVERSIFIED-20260127-$1031.33-5001502951.pdf
  Page 1: 11 detail line(s)
  Page 2: 15 detail line(s)
  Page 3: 7 detail line(s)
  Page 4: 7 detail line(s)
  Page 5: 7 detail line(s)
  Page 6: 7 detail line(s)
  Page 7: 12 detail line(s)
  Done WJF-DIVERSIFIED-20260127-$1031.33-5001502951.pdf: 66 rows
Done. Wrote 66 row(s) to royalty_checks.csv
